# Stage-B v2 checkpoint soup — LB **0.5522**

Публичный notebook без абсолютных путей сервера.

Цепочка: **Stage-A `best.pt`** → Stage-B v2 (human + replay) → soup **`(1-α)·step_400 + α·step_2000`**, best **α=0.15** → submit.

- **pair_text v1** обязателен (не ecup_v2 v2)
- submit **без** symmetry TTA
- init берётся из предыдущего notebook Stage-A (`research_worker_s42_02`)

In [9]:
import json, os, subprocess, sys
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow
import pyarrow.parquet as pq
import sklearn
import torch, transformers

START = Path.cwd().resolve()
masked = lambda path: f"***/{Path(path).name}"

def find_file(name: str, *roots: Path) -> Path:
    for root in roots:
        p = (root / name).resolve()
        if p.exists():
            return p
    raise FileNotFoundError(f"Не найден {name}")

def resolve_script(env_name: str, default: str) -> Path:
    rel = os.getenv(env_name, default)
    p = Path(rel).expanduser()
    if p.is_absolute() and p.exists():
        return p.resolve()
    for base in [START, *START.parents]:
        cand = (base / rel).resolve()
        if cand.exists():
            return cand
    raise FileNotFoundError(f"Не найден {rel}")

TRAIN_SCRIPT = resolve_script("STAGE_B_SCRIPT", "train_bge_stageb_v2.py")
BLEND_SCRIPT = resolve_script("STAGE_B_BLEND_SCRIPT", "final_4_models/scripts/blend_checkpoint_soup.py")
BUILD_SUBMIT = resolve_script("STAGE_B_BUILD_SUBMIT", "notebooks/v2_soup_lb_5522/build_submit.py")

PROJECT_DIR = TRAIN_SCRIPT.parent
DATA_DIR = Path(os.getenv("BGE_DATA_DIR", PROJECT_DIR)).expanduser().resolve()
OUTPUT_ROOT = Path(os.getenv("V2_OUTPUT_ROOT", START / "output")).expanduser().resolve()
LIB = find_file("pair_text_v1.py", START / "../lib", START.parent / "lib", PROJECT_DIR / "notebooks/lib")
sys.path.insert(0, str(LIB.parent))

STAGE_A_RUN = os.getenv("STAGE_A_RUN_ID", "research_worker_s42_02")
STAGE_A_OUT = Path(os.getenv(
    "STAGE_A_OUTPUT",
    START.parent / "initial_stage_a_user_bge" / "output" / STAGE_A_RUN,
)).expanduser().resolve()
STAGE_A_INIT = Path(os.getenv("STAGE_A_INIT", STAGE_A_OUT / "best.pt")).expanduser().resolve()

assert TRAIN_SCRIPT.exists(), "Укажите STAGE_B_SCRIPT"
assert BLEND_SCRIPT.exists(), "Укажите STAGE_B_BLEND_SCRIPT"
assert BUILD_SUBMIT.exists(), "Укажите STAGE_B_BUILD_SUBMIT"
assert STAGE_A_INIT.exists(), f"Нет Stage-A init: ***/{STAGE_A_INIT.name}"

EXPECTED_VERSIONS = {
    "python": "3.12", "torch": "2.6.0+cu124", "transformers": "4.57.6",
    "numpy": "2.2.6", "pandas": "2.3.3", "pyarrow": "23.0.1", "scikit-learn": "1.8.0",
}
ACTUAL_VERSIONS = {
    "python": ".".join(map(str, sys.version_info[:2])),
    "torch": torch.__version__, "transformers": transformers.__version__,
    "numpy": np.__version__, "pandas": pd.__version__,
    "pyarrow": pyarrow.__version__, "scikit-learn": sklearn.__version__,
}
assert ACTUAL_VERSIONS == EXPECTED_VERSIONS, ACTUAL_VERSIONS

print("train script:", masked(TRAIN_SCRIPT))
print("blend script:", masked(BLEND_SCRIPT))
print("data dir: *** | output root: ***")
print("stage-a init:", masked(STAGE_A_INIT), "OK")
print("environment: locked and verified")

rows = []
for name in ["items.parquet", "items_human.parquet", "matches.parquet", "matches_llm.parquet"]:
    path = DATA_DIR / name
    assert path.exists(), f"Нет ***/{name}"
    rows.append({"file": f"***/{name}", "rows": pq.ParquetFile(path).metadata.num_rows,
                 "size_GiB": round(path.stat().st_size / 1024**3, 3)})
display(pd.DataFrame(rows))

train script: ***/train_bge_stageb_v2.py
blend script: ***/blend_checkpoint_soup.py
data dir: *** | output root: ***
stage-a init: ***/best.pt OK
environment: locked and verified


,file,rows,size_GiB
0,***/items.parquet,13397761,3.822
1,***/items_human.parquet,711304,0.199
2,***/matches.parquet,365654,0.004
3,***/matches_llm.parquet,11187780,0.098


In [10]:
from verify_pair_text import compare_v1_vs_v2, main as verify_main
assert verify_main() == 0
cmp = compare_v1_vs_v2()
print(json.dumps({k: v for k, v in cmp.items() if k.endswith("_preview") is False}, ensure_ascii=False, indent=2))
print("OK — pair_text v1 для soup/TTA submit")

pair_text v1 verification
  version=v1 attr_limit=520
  v1 vs v2 equal: False (must be False for fashion stress)
  v2 size-first: True | v1 size-first: False
OK — use pair_text_v1 for v2 soup & symmetry TTA pipelines
{
  "v1_len": 181,
  "v2_len": 181,
  "v1_has_size_first": false,
  "v2_has_size_first": true,
  "texts_equal": false
}
OK — pair_text v1 для soup/TTA submit


In [11]:
RUN_ID = "v2_soup_s42_01"
GPU_IDS = "0,1"   
NPROC = len(GPU_IDS.split(","))
MAX_USED_MIB = 5_000
RUN_TRAIN = True      # Stage-B v2 от нового Stage-A init
RUN_BLEND = True      # soup после train (или если чекпойнты уже есть)
RUN_SUBMIT = True

V2_OUT = OUTPUT_ROOT / RUN_ID / "stageb_v2"
SOUP_OUT = OUTPUT_ROOT / RUN_ID / "soup_run"
LOG = OUTPUT_ROOT / RUN_ID / "stageb_v2.log"
V2_OUT.mkdir(parents=True, exist_ok=True)
SOUP_OUT.mkdir(parents=True, exist_ok=True)

CKPT_A = V2_OUT / "checkpoints/step_00400.pt"
CKPT_B = V2_OUT / "checkpoints/step_02000.pt"

CONFIG = {
    "stage_a_init": f"***/{STAGE_A_INIT.name}",
    "stage_a_holdout": 0.8600,
    "pair_text": "v1",
    "stage_b_out": f"***/{V2_OUT.name}",
    "soup_out": f"***/{SOUP_OUT.name}",
    "blend_formula": "(1-alpha)*step_2000 + alpha*step_400",
    "blend_alphas": "0.15,0.20,0.25,0.30,0.35",
    "physical_gpu_ids": GPU_IDS,
    "nproc": NPROC,
    "run_train": RUN_TRAIN,
    "run_blend": RUN_BLEND,
    "run_submit": RUN_SUBMIT,
}
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

def run_cmd(cmd: str, env: dict | None = None, log_path: Path | None = None) -> int:
    env = {**os.environ, **(env or {})}
    shown = cmd.replace(str(PROJECT_DIR), "***").replace(str(DATA_DIR), "***")
    shown = shown.replace(str(V2_OUT), "***").replace(str(SOUP_OUT), "***").replace(str(STAGE_A_INIT), "***")
    print("$", shown, flush=True)
    if log_path:
        log_path.parent.mkdir(parents=True, exist_ok=True)
        with log_path.open("a", encoding="utf-8") as f:
            f.write(f"$ {shown}\n")
            p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
            assert p.stdout is not None
            for line in p.stdout:
                print(line, end="", flush=True)
                f.write(line)
            return p.wait()
    return subprocess.call(cmd, shell=True, env=env)

def gpu_used_mib(ids: list[int]) -> dict[int, int]:
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=index,memory.used", "--format=csv,noheader,nounits"], text=True
    )
    used = {int(i): int(m) for i, m in (ln.split(",") for ln in out.strip().splitlines())}
    return {i: used[i] for i in ids}

selected = [int(x) for x in GPU_IDS.split(",")]
used = gpu_used_mib(selected)
print("selected physical GPUs:", selected, "| used MiB:", used)
assert all(used[i] <= MAX_USED_MIB for i in selected), f"GPU заняты: {used}"

{
  "stage_a_init": "***/best.pt",
  "stage_a_holdout": 0.86,
  "pair_text": "v1",
  "stage_b_out": "***/stageb_v2",
  "soup_out": "***/soup_run",
  "blend_formula": "(1-alpha)*step_2000 + alpha*step_400",
  "blend_alphas": "0.15,0.20,0.25,0.30,0.35",
  "physical_gpu_ids": "0,1",
  "nproc": 2,
  "run_train": true,
  "run_blend": true,
  "run_submit": true
}
selected physical GPUs: [0, 1] | used MiB: {0: 1907, 1: 1}


In [13]:
STAGE_B_ENV = dict(
    STAGE_B_INIT_CKPT=str(STAGE_A_INIT),
    STAGE_B_OUT_DIR=str(V2_OUT),
    STAGE_B_ITEMS_PATH=str(DATA_DIR / "items.parquet"),
    STAGE_B_ITEMS_HUMAN_PATH=str(DATA_DIR / "items_human.parquet"),
    STAGE_B_MATCHES_HUMAN=str(DATA_DIR / "matches.parquet"),
    STAGE_B_MATCHES_LLM=str(DATA_DIR / "matches_llm.parquet"),
)

if RUN_TRAIN:
    if not (V2_OUT / "teacher_probs.npz").exists():
        rc = run_cmd(
            f"cd {PROJECT_DIR} && CUDA_VISIBLE_DEVICES={GPU_IDS.split(',')[0]} "
            f"python {TRAIN_SCRIPT} --precompute-teacher",
            env=STAGE_B_ENV, log_path=LOG,
        )
        assert rc == 0, "precompute-teacher failed"
    else:
        print("teacher cache already exists — skip precompute")

    rc = run_cmd(
        f"cd {PROJECT_DIR} && CUDA_VISIBLE_DEVICES={GPU_IDS} "
        f"python -m torch.distributed.run --nproc_per_node={NPROC} {TRAIN_SCRIPT}",
        env=STAGE_B_ENV, log_path=LOG,
    )
    assert rc == 0, "Stage-B v2 train failed"
else:
    print("RUN_TRAIN=False — используем существующие checkpoints")

assert CKPT_A.exists() and CKPT_B.exists(), "Нет step_00400 / step_02000 для soup"

$ cd *** && CUDA_VISIBLE_DEVICES=0 python ***/train_bge_stageb_v2.py --precompute-teacher
PRECOMPUTE teacher cache (single GPU)
manual split: {'train': 292428, 'tune': 36609, 'eval': 36617}
loading 805,310 texts...
items_human hit 711,304/805,310
items.parquet scan; missing left 0; total texts 805,310
oversample Обувь: 292,428 -> 307,117 human train rows
gray: full=45,831 sample=11,851
Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at deepvk/USER-bge-m3 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
scoring teacher probs for human train (307,117 pairs)...
  human: 128/307,117 (0.0%) 244 pair/s
  human: 25,728/307,117 (8.4%) 546 pair/s
  human: 51,328/307,117 (16.7%) 547 pair/s
  human: 76,928/307,117 (25.0%) 548 pair/s
  human: 102,528/

In [14]:
if RUN_BLEND:
    scripts_dir = BLEND_SCRIPT.parent
    sys.path.insert(0, str(scripts_dir))
    sys.path.insert(0, str(LIB.parent))
    from verify_pair_text import patch_score_ensemble_v1
    patch_score_ensemble_v1()
    rc = run_cmd(
        f"cd {PROJECT_DIR} && PYTHONPATH={scripts_dir}:{LIB.parent} "
        f"python {BLEND_SCRIPT} --ckpt-a {CKPT_A} --ckpt-b {CKPT_B} "
        f"--out-dir {SOUP_OUT} --alphas 0.15,0.20,0.25,0.30,0.35 --skip-submit --gpu {GPU_IDS.split(',')[0]}",
        env=STAGE_B_ENV,
    )
    assert rc == 0, "blend failed"
else:
    print("RUN_BLEND=False — пропуск soup")

metrics = json.loads((SOUP_OUT / "metrics.json").read_text())
def mask_obj(o):
    if isinstance(o, dict):
        return {k: mask_obj(v) for k, v in o.items()}
    if isinstance(o, list):
        return [mask_obj(x) for x in o]
    if isinstance(o, str) and "/" in o:
        return masked(o)
    return o
print("best alpha:", metrics["blend"]["best_alpha"])
print(json.dumps(mask_obj(metrics["best_metrics"]), ensure_ascii=False, indent=2))

$ cd *** && PYTHONPATH=***/final_4_models/scripts:***/notebooks/lib python ***/final_4_models/scripts/blend_checkpoint_soup.py --ckpt-a ***/notebooks/v2_soup_lb_5522/output/v2_soup_s42_01/stageb_v2/checkpoints/step_00400.pt --ckpt-b ***/notebooks/v2_soup_lb_5522/output/v2_soup_s42_01/stageb_v2/checkpoints/step_02000.pt --out-dir ***/notebooks/v2_soup_lb_5522/output/v2_soup_s42_01/soup_run --alphas 0.15,0.20,0.25,0.30,0.35 --skip-submit --gpu 0
checkpoint soup | A=/home/dgbabenko/assistant-peft/notebooks/v2_soup_lb_5522/output/v2_soup_s42_01/stageb_v2/checkpoints/step_00400.pt | B=/home/dgbabenko/assistant-peft/notebooks/v2_soup_lb_5522/output/v2_soup_s42_01/stageb_v2/checkpoints/step_02000.pt
blend: theta = (1-alpha)*B + alpha*A  | alphas=[0.15, 0.2, 0.25, 0.3, 0.35]
OUT=/home/dgbabenko/assistant-peft/notebooks/v2_soup_lb_5522/output/v2_soup_s42_01/soup_run
loading 141,261 texts...
  items scan: 23,294/141,261 ids (16s)
  items scan: 48,122/141,261 ids (27s)
  items scan: 75,590/141,26

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at deepvk/USER-bge-m3 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  alpha=0.15 (B=0.85): gray=0.5478 problem=0.6333 tune=0.7987 score=0.6407  [150.1s]


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at deepvk/USER-bge-m3 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  alpha=0.20 (B=0.80): gray=0.5481 problem=0.6306 tune=0.7979 score=0.6393  [148.3s]


...

  alpha=0.30 (B=0.70): gray=0.5488 problem=0.6243 tune=0.7960 score=0.6360  [148.2s]


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at deepvk/USER-bge-m3 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  alpha=0.35 (B=0.65): gray=0.5491 problem=0.6210 tune=0.7949 score=0.6342  [148.0s]

BEST alpha=0.15 score=0.6407


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at deepvk/USER-bge-m3 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


saved /home/dgbabenko/assistant-peft/notebooks/v2_soup_lb_5522/output/v2_soup_s42_01/soup_run/best.pt /home/dgbabenko/assistant-peft/notebooks/v2_soup_lb_5522/output/v2_soup_s42_01/soup_run/export_fp16/ metrics.json
best alpha: 0.15
{
  "alpha_step400": 0.15,
  "weight_step2000": 0.85,
  "gray_full": 0.5478190201523339,
  "problem_ap": 0.6332693303378784,
  "tune_macro": 0.7987087541036964,
  "composite_score": 0.6407221220353786,
  "elapsed_s": 150.1
}


In [15]:
if RUN_SUBMIT:
    assert run_cmd(f"python {BUILD_SUBMIT} --run-dir {SOUP_OUT}") == 0
    zip_path = SOUP_OUT / "matching-bge-human-ft-submit.zip"
    print("submit zip:", masked(zip_path), f"{zip_path.stat().st_size/1024**2:.1f} MB")
else:
    print("RUN_SUBMIT=False")

$ python ***/notebooks/v2_soup_lb_5522/build_submit.py --run-dir ***/notebooks/v2_soup_lb_5522/output/v2_soup_s42_01/soup_run
created /home/dgbabenko/assistant-peft/notebooks/v2_soup_lb_5522/output/v2_soup_s42_01/soup_run/matching-bge-human-ft-submit.zip (1268.8 MB)
sha256: 908f60451f01463d7574e488c4655bea2b52bbac7fa19be43842a8e0a3ee5260
submit zip: ***/matching-bge-human-ft-submit.zip 1268.8 MB


In [ ]:
import zipfile, tempfile
zip_path = SOUP_OUT / "matching-bge-human-ft-submit.zip"
assert zip_path.exists(), "Нет submit zip"
with tempfile.TemporaryDirectory() as td:
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(td)
    root = next(Path(td).glob("matching-bge-human-ft*"))
    sys.path.insert(0, str(root))
    os.chdir(root)
    from src.utils import build_text
    print("import OK:", build_text("test", '{"Бренд":"X"}')[:40])

if (V2_OUT / "metrics.json").exists():
    stageb = json.loads((V2_OUT / "metrics.json").read_text())
    print("Stage-B metrics keys:", list(stageb.keys())[:8])
print("Готово: soup + submit в output run dir (пути скрыты в логах)")